# Broadcast Hash Joins: Eliminating Network Shuffles

**Module**: 06_memory_and_storage  
**Notebook**: 07_broadcast_join.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Broadcast Hash Join (BHJ) works, how to force it using broadcast(), and why it is the fastest join in distributed computing.
- **Why does this topic exist?**: BHJ completely eliminates the network shuffle and disk sort phases for the large table.
- **What problem does it solve?**: Running slow Sort-Merge Joins on small dimension tables.
- **Where is it used in real-world Data Engineering?**: Star-schema fact-dimension joins in production ETL.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Joins, broadcast variables, AQE |
| **Required depth** | Working knowledge: can recognize and tune a broadcast hash join |
| **Data Engineer** | Use broadcast joins safely and diagnose when the build side is too large. |
| **Senior Data Engineer** | Choose between broadcast and sort-merge based on size, skew, and memory. |
| **Spark specialist** | Reason about broadcast thresholds, hints, AQE conversion, and executor memory. |
| **Next topic** | `07_spark_sql_internals/01_catalyst_optimizer.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A broadcast hash join sends the small side to every executor and avoids shuffling the large side. It can turn a
multi-hour shuffle into a fast map-side join, but it can also exhaust executor memory when the "small" side grows.
The optimizer chooses it automatically below a size threshold, and AQE can convert a sort-merge join at runtime.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Broadcast hash join** | Replicate small side, stream large side | No shuffle of the large table |
| **Build side** | The replicated side | Must fit in executor memory |
| **Probe side** | The streamed side | Read partition by partition |
| **Broadcast threshold** | `spark.sql.autoBroadcastJoinThreshold` | Default 10 MB; -1 disables automatic broadcast |
| **Broadcast hint** | `/*+ BROADCAST(t) */` | Requests a broadcast join |
| **AQE conversion** | Runtime plan change | Can convert sort-merge to broadcast when statistics justify it |

### How it works, step by step

1. The optimizer estimates the size of each join side.
2. If one side is below the broadcast threshold, it plans a BroadcastHashJoin.
3. The driver collects and broadcasts the build side.
4. Each executor builds a hash table from the broadcast data.
5. The probe side is streamed partition by partition and matched against the hash table.
6. If the build side is too large, executors may spill or fail with memory errors.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Broadcast Hash Join Architecture

```
Large Table (Fact):       [P0]    [P1]    [P2]    [P3] (Zero Shuffle!)
                            |         |         |         |
Small Table (Dimension):  [Broadcast to All Executors] (Single Network Copy)
                            |         |         |         |
Execution:                Local in-memory hash join on each core!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Broadcast_Join_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")

# Force Broadcast Hash Join
bhj_df = orders.join(F.broadcast(products), on="product_id")
print("=== BroadcastHashJoin Physical Plan ===")
bhj_df.explain("formatted")

---

## Experiment: BHJ vs Sort-Merge Join Benchmark

- **1. What are we testing?**: Measure execution latency between BHJ and Sort-Merge Join on retail dataset.
- **2. What do we expect?**: BHJ runs significantly faster because orders table requires zero network shuffle.

In [ ]:
# 1. Force Sort-Merge Join
t0 = time.time()
orders.join(products.hint("merge"), "product_id").write.mode("overwrite").format("noop").save()
t_smj = time.time() - t0

# 2. Force Broadcast Hash Join
t0 = time.time()
orders.join(F.broadcast(products), "product_id").write.mode("overwrite").format("noop").save()
t_bhj = time.time() - t0

print(f"Sort-Merge Join time:   {t_smj:.3f}s (Shuffles both tables)")
print(f"Broadcast Hash Join:    {t_bhj:.3f}s (Zero shuffle on orders)")
print(f"Speedup:                {t_smj / t_bhj:.2f}x")

**3. What happened?**: BHJ finished much faster with zero shuffle on the large orders dataset.  
**4. Why did it happen?**: Orders partitions were processed locally without network transfer or disk sorting.  
**5. Practical Engineering Lesson**: Always broadcast lookup/dimension tables when they fit in executor memory.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Enriching events with a small dimension

- **Industry**: E-commerce
- **Scenario**: Billions of click events are enriched with a product catalog of a few megabytes.
- **Problem**: A sort-merge join would shuffle the entire event stream.
- **Why this concept is relevant**: Broadcasting the catalog avoids the shuffle and lets each executor enrich its partition locally.
- **Implementation idea**: Confirm the catalog size, use a broadcast hint if needed, and monitor executor memory.
- **Expected result**: A join with no large-side shuffle and much lower latency.
- **Alternative**: A sort-merge join with pre-partitioned data.
- **Trade-offs**: Broadcast is faster when the dimension fits; sort-merge scales to large-large joins.

### Use Case 2: The dimension that outgrew the threshold

- **Industry**: Telecom
- **Scenario**: A subscriber dimension grows from 5 MB to 2 GB, but the job still forces a broadcast hint.
- **Problem**: Executors run out of memory while building the hash table.
- **Why this concept is relevant**: The broadcast threshold exists to protect memory; forcing a broadcast beyond it is unsafe.
- **Implementation idea**: Remove the hint, let the optimizer choose sort-merge, or pre-aggregate/filter the dimension.
- **Expected result**: A stable join that scales with the dimension.
- **Alternative**: Increase executor memory and keep the broadcast.
- **Trade-offs**: More memory may postpone the failure; sort-merge is the scalable default for large-large joins.

---

## Business Value

- Lower latency for dimension enrichment.
- Less network traffic and shuffle I/O.
- Lower cost for common star-schema joins.
- Clear criteria for when to broadcast.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Check the actual build-side size, not the table's total size on disk.
- Use a broadcast hint only when you have verified the size.
- Monitor executor memory and spill after enabling broadcast.
- Prefer sort-merge for large-large joins.
- Remember that AQE can change the join strategy at runtime.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Find the broadcast threshold

**Steps**

1. Join a large fact table to dimensions of increasing size.
2. Inspect the plan for BroadcastHashJoin versus SortMergeJoin.
3. Record executor memory and runtime around the threshold.

**What to observe**: The optimizer switches strategies based on estimated size; forcing broadcast beyond the threshold risks OOM.

**Healthy result**: The chosen strategy matches the data size and memory budget.

**Problematic result**: A forced broadcast causes spill or executor failure.

**Engineering takeaway**: Broadcast is a memory trade, not a universal optimization.

---

## Trade-offs and Alternatives

### Advantages

- Avoids large-side shuffle
- Fast for star schemas
- Optimizer support

### Limitations

- Build side must fit in memory
- Can cause executor OOM
- Broadcast is repeated per executor

### When to use it

- Dimension enrichment
- Small lookup joins
- Star-schema queries

### When not to use it

- Forcing broadcast on large tables
- Broadcasting without size checks
- Ignoring executor memory

### Alternatives inside Spark

- BroadcastHashJoin
- AQE
- Join hints

### Alternatives outside Spark

- Warehouse broadcast joins
- Distributed lookup services
- Pre-joined tables

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Auto-Broadcast Threshold
**Problem Statement**: Print spark.sql.autoBroadcastJoinThreshold.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Disable Auto-Broadcast
**Problem Statement**: Set threshold to -1 and observe join plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Detect BroadcastExchange Node
**Problem Statement**: Locate BroadcastExchange in physical plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Broadcast join with a misleading size estimate

**Scenario**: The optimizer chooses a broadcast join, but the build side is much larger after filtering than the catalog statistics suggest.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Run `ANALYZE TABLE` or check the statistics used by the optimizer.
2. Inspect the actual build-side row count and size.
3. Disable broadcast or add a sort-merge hint if the build side is too large.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Auto-Broadcast Threshold):
```python
print(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')) # Default 10485760 (10 MB)
```

#### Solution 2 (Disable Auto-Broadcast):
```python
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','-1')
orders.join(products,'product_id').explain()
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','10485760')
```

#### Solution 3 (Detect BroadcastExchange Node):
```python
print('BroadcastExchangeExec found in plan')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Fastest Join**: Broadcast Hash Join eliminates fact table shuffle.
- **Threshold**: Default 10 MB (`autoBroadcastJoinThreshold`).
- **Hint**: `F.broadcast(df)` or `df.hint('broadcast')`.

### Essential Technical Terms
- `BroadcastHashJoinExec`: Physical operator for broadcast join.
- `BroadcastExchange`: Collects table to driver and broadcasts to executors.

### Common Engineering Mistakes
- **Broadcasting a massive table (>2 GB)**: Causes Driver OOM during broadcast collection.

### Interview Questions on This Topic
- **Q: What are the requirements for a Broadcast Hash Join?**
  - *A*: 1. Equi-join condition (=), and 2. One table small enough to fit in driver and executor memory.


### More Interview Questions

- **Q: When does Spark choose a broadcast hash join?**
  - *A*: When one side's estimated size is below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default) or when a broadcast hint is used, and the join is eligible.
- **Q: What is the main risk of a broadcast join?**
  - *A*: The build side is replicated to every executor; if it is larger than expected, executors can spill or run out of memory.

### What to Study Next
Module 06 complete! Proceed to Module 07: Spark SQL Internals.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')